# 🏠 Projet Complet Data Science : Analyse Prédictive Immobilière CI

## Bienvenue dans ce projet pédagogique de Data Science

Ce notebook vous guidera à travers **toutes les étapes d'un projet data science complet** appliqué à la gestion immobilière ivoirienne.

### 📋 Plan du cours :

1. **📊 Exploration et Nettoyage des Données (EDA)**
2. **📈 Visualisation et Analyse Descriptive**
3. **🏠 Analyse des Prix par Commune et Type de Logement**
4. **💰 Prédiction des Loyers (Machine Learning - Régression)**
5. **📉 Détection d'Anomalies dans les Paiements**
6. **🎯 Segmentation des Locataires (Clustering)**
7. **📊 Dashboard Interactif et Conclusions**

---

**Objectifs pédagogiques :**
- Maîtriser le workflow complet d'un projet data science
- Apprendre les bibliothèques essentielles (pandas, numpy, matplotlib, seaborn, scikit-learn)
- Comprendre comment appliquer le ML à un cas réel
- Développer votre intuition analytique

## 🚀 Partie 1 : Importation des Bibliothèques et Génération des Données

In [ ]:
# Importation des bibliothèques essentielles
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliothèques importées avec succès !")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

### 📝 Note importante sur les données

Dans un projet réel, vous chargeriez vos données depuis une base de données ou des fichiers CSV. Pour ce cours, nous allons **générer des données réalistes** basées sur le contexte ivoirien.

In [ ]:
# ============================================================
# GÉNÉRATION DE DONNÉES RÉALISTES
# ============================================================

# Configuration des paramètres ivoiriens
communes = [
    'Cocody', 'Yopougon', 'Plateau', 'Marcory', 'Treichville',
    'Adjamé', 'Abobo', 'Koumassi', 'Port-Bouët', 'Attécoubé',
    'Bingerville', 'Songon', 'Anyama'
]

types_logement = ['Studio', 'Appartement', 'Villa', 'Duplex', 'Chambre', 'Local commercial']

modes_paiement = ['Orange Money', 'MTN MoMo', 'Wave', 'Espèces', 'Chèque', 'Virement bancaire']

types_paiement = ['Loyer', 'Caution', 'Électricité (CIE)', 'Eau (SODECI)', 'Frais d\'entretien']

# Prix de base par commune (en FCFA)
prix_base_commune = {
    'Plateau': 500000,
    'Cocody': 400000,
    'Marcory': 300000,
    'Treichville': 200000,
    'Yopougon': 150000,
    'Adjamé': 140000,
    'Abobo': 120000,
    'Koumassi': 130000,
    'Port-Bouët': 180000,
    'Attécoubé': 135000,
    'Bingerville': 160000,
    'Songon': 100000,
    'Anyama': 110000
}

# Multiplicateur par type de logement
multiplicateur_type = {
    'Chambre': 0.5,
    'Studio': 0.8,
    'Appartement': 1.0,
    'Duplex': 1.8,
    'Villa': 2.5,
    'Local commercial': 1.5
}

def generer_donnees_proprietes(n=200):
    """Génère des données de propriétés réalistes"""
    proprietes = []
    
    for i in range(n):
        commune = random.choice(communes)
        type_logement = random.choice(types_logement)
        
        # Calcul du prix basé sur la commune et le type
        prix_base = prix_base_commune[commune] * multiplicateur_type[type_logement]
        
        # Variation aléatoire (±20%)
        prix_final = prix_base * random.uniform(0.8, 1.2)
        
        # Nombre de pièces selon le type
        if type_logement == 'Chambre':
            nb_pieces = 1
        elif type_logement == 'Studio':
            nb_pieces = 1
        elif type_logement == 'Appartement':
            nb_pieces = random.randint(2, 5)
        elif type_logement in ['Villa', 'Duplex']:
            nb_pieces = random.randint(4, 8)
        else:
            nb_pieces = random.randint(1, 3)
        
        # Superficie estimée
        superficie = nb_pieces * random.uniform(25, 40)
        
        propriete = {
            'id': i + 1,
            'titre': f"{type_logement} à {commune}",
            'type': type_logement,
            'commune': commune,
            'nb_pieces': nb_pieces,
            'superficie': round(superficie, 2),
            'prix_loyer': round(prix_final, 0),
            'statut': random.choices(['Disponible', 'Loué'], weights=[0.3, 0.7])[0],
            'date_creation': datetime.now() - timedelta(days=random.randint(0, 365))
        }
        proprietes.append(propriete)
    
    return pd.DataFrame(proprietes)

def generer_donnees_locataires(n=150, proprietes_df=None):
    """Génère des données de locataires réalistes"""
    prenoms = ['Kouamé', 'Koné', 'Traoré', 'Diabaté', 'Ouattara', 'Coulibaly', 
               'Bamba', 'Dosso', 'Camara', 'Sanogo', 'Touré', 'Fofana',
               'Marie', 'Fatou', 'Aminata', 'Sarah', 'Christine', 'Brigitte']
    noms = ['Jean', 'Pierre', 'Michel', 'Paul', 'Jacques', 'François', 
            'Alain', 'Roger', 'Philippe', 'Daniel', 'Patrick', 'Christian']
    professions = ['Enseignant', 'Médecin', 'Commerçant', 'Ingénieur', 
                   'Banquier', 'Juriste', 'Entrepreneur', 'Fonctionnaire',
                   'Infirmier', 'Architecte', 'Comptable', 'Journaliste']
    
    locataires = []
    
    for i in range(n):
        locataire = {
            'id': i + 1,
            'nom': random.choice(noms),
            'prenom': random.choice(prenoms),
            'cni': f"{random.randint(10, 99)}{random.randint(100000, 999999)}{random.choice(['A', 'B'])}",
            'telephone': f"+225 0{random.randint(1, 9)} {random.randint(10, 99)} {random.randint(10, 99)} {random.randint(10, 99)}",
            'email': f"locataire{i+1}@gmail.com",
            'profession': random.choice(professions),
            'date_entree': datetime.now() - timedelta(days=random.randint(30, 730)),
            'propriete_id': random.randint(1, len(proprietes_df)) if proprietes_df is not None else None
        }
        locataires.append(locataire)
    
    return pd.DataFrame(locataires)

def generer_donnees_paiements(n=500, locataires_df=None, proprietes_df=None):
    """Génère des données de paiements réalistes avec quelques anomalies"""
    paiements = []
    
    for i in range(n):
        # Sélectionner un locataire et sa propriété
        locataire_id = random.randint(1, len(locataires_df)) if locataires_df is not None else random.randint(1, 150)
        propriete_id = random.randint(1, len(proprietes_df)) if proprietes_df is not None else random.randint(1, 200)
        
        # Récupérer le prix de référence si possible
        if proprietes_df is not None:
            prix_ref = proprietes_df.loc[proprietes_df['id'] == propriete_id, 'prix_loyer'].values
            prix_ref = prix_ref[0] if len(prix_ref) > 0 else 150000
        else:
            prix_ref = 150000
        
        type_paiement = random.choice(types_paiement)
        
        # Montant basé sur le type de paiement
        if type_paiement == 'Loyer':
            montant = prix_ref * random.uniform(0.9, 1.1)
        elif type_paiement == 'Caution':
            montant = prix_ref * random.uniform(1.5, 3.0)
        elif type_paiement in ['Électricité (CIE)', 'Eau (SODECI)']:
            montant = random.uniform(5000, 50000)
        else:
            montant = random.uniform(10000, 100000)
        
        # Ajouter quelques anomalies (5% des cas)
        if random.random() < 0.05:
            montant = montant * random.uniform(2, 5)  # Paiement anormalement élevé
        
        # Date de paiement
        date_paiement = datetime.now() - timedelta(days=random.randint(0, 365))
        
        paiement = {
            'id': i + 1,
            'locataire_id': locataire_id,
            'propriete_id': propriete_id,
            'montant': round(montant, 0),
            'type_paiement': type_paiement,
            'mode_paiement': random.choice(modes_paiement),
            'reference_transaction': f"TXN{random.randint(100000, 999999)}",
            'date_paiement': date_paiement,
            'date_echeance': date_paiement + timedelta(days=30)
        }
        paiements.append(paiement)
    
    return pd.DataFrame(paiements)

# Générer les datasets
print("🔄 Génération des données...")
df_proprietes = generer_donnees_proprietes(200)
df_locataires = generer_donnees_locataires(150, df_proprietes)
df_paiements = generer_donnees_paiements(500, df_locataires, df_proprietes)

print(f"✅ {len(df_proprietes)} propriétés générées")
print(f"✅ {len(df_locataires)} locataires générés")
print(f"✅ {len(df_paiements)} paiements générés")

# Sauvegarder pour utilisation future
df_proprietes.to_csv('/workspace/ci_immobilier/data_science/proprietes.csv', index=False)
df_locataires.to_csv('/workspace/ci_immobilier/data_science/locataires.csv', index=False)
df_paiements.to_csv('/workspace/ci_immobilier/data_science/paiements.csv', index=False)

print("\n📁 Données sauvegardées dans /data_science/")

## 📊 Partie 2 : Exploration et Nettoyage des Données (EDA)

In [ ]:
# Chargement des données (dans un projet réel, vous chargeriez depuis CSV ou BDD)
# df_proprietes = pd.read_csv('proprietes.csv')
# df_locataires = pd.read_csv('locataires.csv')
# df_paiements = pd.read_csv('paiements.csv')

# Affichage des premières lignes
print("=" * 60)
print("📋 APERÇU DES DONNÉES PROPRIÉTÉS")
print("=" * 60)
display(df_proprietes.head())

print("\n" + "=" * 60)
print("📋 APERÇU DES DONNÉES PAIEMENTS")
print("=" * 60)
display(df_paiements.head())

In [ ]:
# Informations générales sur les datasets
print("\n" + "=" * 60)
print("ℹ️  INFORMATIONS SUR LES DATASETS")
print("=" * 60)

print("\n🏠 PROPRIÉTÉS:")
print(f"   - Nombre de lignes: {df_proprietes.shape[0]}")
print(f"   - Nombre de colonnes: {df_proprietes.shape[1]}")
print(f"   - Colonnes: {list(df_proprietes.columns)}")

print("\n💰 PAIEMENTS:")
print(f"   - Nombre de lignes: {df_paiements.shape[0]}")
print(f"   - Nombre de colonnes: {df_paiements.shape[1]}")
print(f"   - Colonnes: {list(df_paiements.columns)}")

print("\n👥 LOCATAIRES:")
print(f"   - Nombre de lignes: {df_locataires.shape[0]}")
print(f"   - Nombre de colonnes: {df_locataires.shape[1]}")

In [ ]:
# Vérification des valeurs manquantes
print("=" * 60)
print("🔍 VÉRIFICATION DES VALEURS MANQUANTES")
print("=" * 60)

missing_proprietes = df_proprietes.isnull().sum()
missing_paiements = df_paiements.isnull().sum()

print("\n🏠 Propriétés - Valeurs manquantes:")
for col, count in missing_proprietes.items():
    percentage = (count / len(df_proprietes)) * 100
    print(f"   {col}: {count} ({percentage:.2f}%)")

print("\n💰 Paiements - Valeurs manquantes:")
for col, count in missing_paiements.items():
    percentage = (count / len(df_paiements)) * 100
    print(f"   {col}: {count} ({percentage:.2f}%)")

if missing_proprietes.sum() == 0 and missing_paiements.sum() == 0:
    print("\n✅ Aucune valeur manquante détectée !")

In [ ]:
# Statistiques descriptives
print("=" * 60)
print("📈 STATISTIQUES DESCRIPTIVES - PROPRIÉTÉS")
print("=" * 60)
display(df_proprietes.describe())

print("\n" + "=" * 60)
print("📈 STATISTIQUES DESCRIPTIVES - PAIEMENTS")
print("=" * 60)
display(df_paiements.describe())

## 📊 Partie 3 : Visualisation et Analyse Descriptive

In [ ]:
# Distribution des prix des loyers
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribution des prix
axes[0, 0].hist(df_proprietes['prix_loyer'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_xlabel('Prix du Loyer (FCFA)')
axes[0, 0].set_ylabel('Nombre de propriétés')
axes[0, 0].set_title('Distribution des Prix des Loyers')
axes[0, 0].ticklabel_format(style='plain', axis='x')

# 2. Nombre de propriétés par commune
commune_counts = df_proprietes['commune'].value_counts()
axes[0, 1].bar(range(len(commune_counts)), commune_counts.values, color='coral', alpha=0.7)
axes[0, 1].set_xticks(range(len(commune_counts)))
axes[0, 1].set_xticklabels(commune_counts.index, rotation=45, ha='right')
axes[0, 1].set_xlabel('Commune')
axes[0, 1].set_ylabel('Nombre de propriétés')
axes[0, 1].set_title('Répartition des Propriétés par Commune')

# 3. Types de logements
type_counts = df_proprietes['type'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(type_counts)))
axes[1, 0].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%', colors=colors)
axes[1, 0].set_title('Distribution des Types de Logements')

# 4. Statut des propriétés
statut_counts = df_proprietes['statut'].value_counts()
axes[1, 1].bar(statut_counts.index, statut_counts.values, color=['lightgreen', 'salmon'], alpha=0.7)
axes[1, 1].set_xlabel('Statut')
axes[1, 1].set_ylabel('Nombre')
axes[1, 1].set_title('Propriétés Disponibles vs Louées')
for i, v in enumerate(statut_counts.values):
    axes[1, 1].text(i, v + 2, str(v), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Analyse des paiements
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribution des montants de paiement
axes[0, 0].hist(df_paiements['montant'], bins=50, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 0].set_xlabel('Montant (FCFA)')
axes[0, 0].set_ylabel('Fréquence')
axes[0, 0].set_title('Distribution des Montants de Paiement')
axes[0, 0].ticklabel_format(style='plain', axis='x')

# 2. Types de paiement
type_paiement_counts = df_paiements['type_paiement'].value_counts()
axes[0, 1].barh(type_paiement_counts.index, type_paiement_counts.values, color='steelblue', alpha=0.7)
axes[0, 1].set_xlabel('Nombre de paiements')
axes[0, 1].set_ylabel('Type de paiement')
axes[0, 1].set_title('Répartition par Type de Paiement')

# 3. Modes de paiement
mode_paiement_counts = df_paiements['mode_paiement'].value_counts()
colors = plt.cm.Pastel1(np.linspace(0, 1, len(mode_paiement_counts)))
axes[1, 0].pie(mode_paiement_counts.values, labels=mode_paiement_counts.index, autopct='%1.1f%%', colors=colors)
axes[1, 0].set_title('Modes de Paiement Utilisés')

# 4. Évolution temporelle des paiements
df_paiements['mois'] = pd.to_datetime(df_paiements['date_paiement']).dt.to_period('M')
paiements_par_mois = df_paiements.groupby('mois')['montant'].sum()
axes[1, 1].plot(range(len(paiements_par_mois)), paiements_par_mois.values, marker='o', linewidth=2, markersize=6)
axes[1, 1].set_xlabel('Mois')
axes[1, 1].set_ylabel('Montant Total (FCFA)')
axes[1, 1].set_title('Évolution des Paiements dans le Temps')
axes[1, 1].ticklabel_format(style='plain', axis='y')
axes[1, 1].set_xticks(range(len(paiements_par_mois)))
axes[1, 1].set_xticklabels([str(m) for m in paiements_par_mois.index], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 🏠 Partie 4 : Analyse des Prix par Commune et Type de Logement

In [ ]:
# Prix moyen par commune
prix_par_commune = df_proprietes.groupby('commune')['prix_loyer'].agg(['mean', 'median', 'min', 'max', 'count']).round(0)
prix_par_commune.columns = ['Prix Moyen', 'Prix Médian', 'Min', 'Max', 'Nombre']
prix_par_commune = prix_par_commune.sort_values('Prix Moyen', ascending=False)

print("=" * 80)
print("💰 PRIX MOYEN DES LOYERS PAR COMMUNE (en FCFA)")
print("=" * 80)
display(prix_par_commune.style.format({'Prix Moyen': '{:,.0f}', 'Prix Médian': '{:,.0f}', 'Min': '{:,.0f}', 'Max': '{:,.0f}'}))

In [ ]:
# Visualisation des prix par commune
plt.figure(figsize=(14, 7))
commune_order = prix_par_commune.index
sns.boxplot(data=df_proprietes, x='commune', y='prix_loyer', order=commune_order, palette='viridis')
plt.xlabel('Commune')
plt.ylabel('Prix du Loyer (FCFA)')
plt.title('Distribution des Prix par Commune')
plt.xticks(rotation=45, ha='right')
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Prix moyen par type de logement
prix_par_type = df_proprietes.groupby('type')['prix_loyer'].agg(['mean', 'median', 'min', 'max', 'count']).round(0)
prix_par_type.columns = ['Prix Moyen', 'Prix Médian', 'Min', 'Max', 'Nombre']
prix_par_type = prix_par_type.sort_values('Prix Moyen', ascending=False)

print("=" * 80)
print("💰 PRIX MOYEN DES LOYERS PAR TYPE DE LOGEMENT (en FCFA)")
print("=" * 80)
display(prix_par_type.style.format({'Prix Moyen': '{:,.0f}', 'Prix Médian': '{:,.0f}', 'Min': '{:,.0f}', 'Max': '{:,.0f}'}))

# Visualisation
plt.figure(figsize=(12, 6))
sns.barplot(data=df_proprietes, x='type', y='prix_loyer', estimator='mean', palette='coolwarm', errorbar='sd')
plt.xlabel('Type de Logement')
plt.ylabel('Prix Moyen (FCFA)')
plt.title('Prix Moyen par Type de Logement')
plt.xticks(rotation=45, ha='right')
plt.ticklabel_format(style='plain', axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap des prix par commune et type
pivot_prix = df_proprietes.pivot_table(values='prix_loyer', index='commune', columns='type', aggfunc='mean')

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_prix, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, 
            cbar_kws={'label': 'Prix Moyen (FCFA)'})
plt.xlabel('Type de Logement')
plt.ylabel('Commune')
plt.title('Carte Thermique des Prix Moyens par Commune et Type')
plt.tight_layout()
plt.show()

## 💰 Partie 5 : Prédiction des Loyers (Machine Learning - Régression)

In [ ]:
# Préparation des données pour le Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Copie des données pour transformation
df_ml = df_proprietes.copy()

# Encodage des variables catégorielles
le_commune = LabelEncoder()
le_type = LabelEncoder()

df_ml['commune_encoded'] = le_commune.fit_transform(df_ml['commune'])
df_ml['type_encoded'] = le_type.fit_transform(df_ml['type'])

# Sélection des features
features = ['commune_encoded', 'type_encoded', 'nb_pieces', 'superficie']
target = 'prix_loyer'

X = df_ml[features]
y = df_ml[target]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Dataset prêt pour le ML!")
print(f"   - Training set: {len(X_train)} échantillons")
print(f"   - Test set: {len(X_test)} échantillons")
print(f"   - Features: {features}")

In [ ]:
# Entraînement et évaluation de plusieurs modèles
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.1),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []

print("=" * 80)
print("🤖 ENTRAÎNEMENT ET ÉVALUATION DES MODÈLES")
print("=" * 80)

for name, model in models.items():
    # Entraînement
    model.fit(X_train, y_train)
    
    # Prédictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Métriques
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)
    
    results.append({
        'Modèle': name,
        'RMSE Train': rmse_train,
        'RMSE Test': rmse_test,
        'MAE Test': mae_test,
        'R² Test': r2_test
    })
    
    print(f"\n{name}:")
    print(f"   RMSE Train: {rmse_train:,.0f} FCFA")
    print(f"   RMSE Test:  {rmse_test:,.0f} FCFA")
    print(f"   MAE Test:   {mae_test:,.0f} FCFA")
    print(f"   R² Test:    {r2_test:.4f}")

# DataFrame des résultats
df_results = pd.DataFrame(results)
print("\n" + "=" * 80)
print("📊 COMPARAISON DES MODÈLES")
print("=" * 80)
display(df_results.sort_values('R² Test', ascending=False).style.format({
    'RMSE Train': '{:,.0f}',
    'RMSE Test': '{:,.0f}',
    'MAE Test': '{:,.0f}',
    'R² Test': '{:.4f}'
})))

In [ ]:
# Visualisation des performances
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Comparaison R²
df_results_sorted = df_results.sort_values('R² Test', ascending=True)
colors = ['red' if x < 0.5 else 'orange' if x < 0.7 else 'green' for x in df_results_sorted['R² Test']]
axes[0].barh(df_results_sorted['Modèle'], df_results_sorted['R² Test'], color=colors, alpha=0.7)
axes[0].set_xlabel('R² Score')
axes[0].set_title('Comparaison des Modèles (R²)')
axes[0].axvline(x=0.7, color='green', linestyle='--', label='Bon seuil (0.7)')
axes[0].legend()

# Comparaison RMSE
axes[1].barh(df_results_sorted['Modèle'], df_results_sorted['RMSE Test'], color='steelblue', alpha=0.7)
axes[1].set_xlabel('RMSE (FCFA)')
axes[1].set_title('Comparaison des Modèles (RMSE)')
axes[1].ticklabel_format(style='plain', axis='x')

plt.tight_layout()
plt.show()

In [ ]:
# Analyse des prédictions du meilleur modèle (Random Forest)
best_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("=" * 80)
print("🎯 IMPORTANCE DES FEATURES (Random Forest)")
print("=" * 80)
display(feature_importance.style.format({'Importance': '{:.4f}'}))

# Visualisation
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='coral', alpha=0.7)
plt.xlabel('Importance')
plt.title('Importance des Variables dans la Prédiction des Prix')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Graphique des prédictions vs valeurs réelles
plt.figure(figsize=(10, 10))
plt.scatter(y_test, y_pred, alpha=0.6, s=50, color='blue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Parfaite prédiction')
plt.xlabel('Prix Réels (FCFA)')
plt.ylabel('Prix Prédits (FCFA)')
plt.title('Prédictions vs Valeurs Réelles (Random Forest)')
plt.legend()
plt.ticklabel_format(style='plain', axis='both')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Résidus
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.6, color='purple')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Prix Prédits (FCFA)')
plt.ylabel('Résidus (FCFA)')
plt.title('Analyse des Résidus')
plt.ticklabel_format(style='plain', axis='both')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 📉 Partie 6 : Détection d'Anomalies dans les Paiements

In [ ]:
# Méthode statistique simple (IQR)
Q1 = df_paiements['montant'].quantile(0.25)
Q3 = df_paiements['montant'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

anomalies_iqr = df_paiements[(df_paiements['montant'] < lower_bound) | (df_paiements['montant'] > upper_bound)]

print("=" * 80)
print("🔍 DÉTECTION D'ANOMALIES - MÉTHODE IQR")
print("=" * 80)
print(f"\nSeuils statistiques:")
print(f"   Q1 (25ème percentile): {Q1:,.0f} FCFA")
print(f"   Q3 (75ème percentile): {Q3:,.0f} FCFA")
print(f"   IQR: {IQR:,.0f} FCFA")
print(f"   Borne inférieure: {lower_bound:,.0f} FCFA")
print(f"   Borne supérieure: {upper_bound:,.0f} FCFA")
print(f"\n🚨 Nombre d'anomalies détectées: {len(anomalies_iqr)}")
print(f"   Pourcentage: {(len(anomalies_iqr) / len(df_paiements)) * 100:.2f}%")

if len(anomalies_iqr) > 0:
    print("\n📋 Top 10 des anomalies:")
    display(anomalies_iqr.nlargest(10, 'montant')[['id', 'montant', 'type_paiement', 'mode_paiement', 'date_paiement']].style.format({'montant': '{:,.0f}'}))

In [ ]:
# Isolation Forest (Machine Learning)
from sklearn.ensemble import IsolationForest

# Préparation des données
df_anomaly = df_paiements[['montant']].copy()

# Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df_anomaly['anomaly_score'] = iso_forest.fit_predict(df_anomaly[['montant']])
df_anomaly['is_anomaly'] = df_anomaly['anomaly_score'].apply(lambda x: 'Anomalie' if x == -1 else 'Normal')

# Ajouter au dataframe original
df_paiements['anomaly_flag'] = df_anomaly['is_anomaly']

anomalies_ml = df_paiements[df_paiements['anomaly_flag'] == 'Anomalie']

print("=" * 80)
print("🔍 DÉTECTION D'ANOMALIES - ISOLATION FOREST")
print("=" * 80)
print(f"\n🚨 Nombre d'anomalies détectées: {len(anomalies_ml)}")
print(f"   Pourcentage: {(len(anomalies_ml) / len(df_paiements)) * 100:.2f}%")

if len(anomalies_ml) > 0:
    print("\n📋 Top 10 des anomalies:")
    display(anomalies_ml.nlargest(10, 'montant')[['id', 'montant', 'type_paiement', 'mode_paiement', 'date_paiement']].style.format({'montant': '{:,.0f}'}))

In [ ]:
# Visualisation des anomalies
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot avec anomalies
normal_payments = df_paiements[df_paiements['anomaly_flag'] == 'Normal']

axes[0].boxplot([normal_payments['montant'], anomalies_ml['montant']], 
                labels=['Paiements Normaux', 'Anomalies'],
                patch_artist=True)
axes[0].set_ylabel('Montant (FCFA)')
axes[0].set_title('Comparaison: Paiements Normaux vs Anomalies')
axes[0].ticklabel_format(style='plain', axis='y')

# Scatter plot temporel
df_paiements_sorted = df_paiements.sort_values('date_paiement')
colors = ['green' if x == 'Normal' else 'red' for x in df_paiements_sorted['anomaly_flag']]

axes[1].scatter(range(len(df_paiements_sorted)), df_paiements_sorted['montant'], 
               c=colors, alpha=0.6, s=30)
axes[1].axhline(y=upper_bound, color='red', linestyle='--', label='Seuil anomalie')
axes[1].set_xlabel('Index (trié par date)')
axes[1].set_ylabel('Montant (FCFA)')
axes[1].set_title('Détection d\'Anomalies dans le Temps')
axes[1].legend()
axes[1].ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

## 🎯 Partie 7 : Segmentation des Locataires (Clustering)

In [ ]:
# Agrégation des données par locataire
df_locataire_stats = df_paiements.groupby('locataire_id').agg({
    'montant': ['sum', 'mean', 'count'],
    'date_paiement': ['min', 'max']
}).reset_index()

df_locataire_stats.columns = ['locataire_id', 'montant_total', 'montant_moyen', 'nb_paiements', 
                              'premier_paiement', 'dernier_paiement']

# Calcul de l'ancienneté
df_locataire_stats['anciennete_jours'] = (pd.to_datetime(df_locataire_stats['dernier_paiement']) - 
                                          pd.to_datetime(df_locataire_stats['premier_paiement'])).dt.days

# Fusion avec les infos locataires
df_clustering = df_locataire_stats.merge(df_locataires, left_on='locataire_id', right_on='id', how='left')

print("✅ Données préparées pour le clustering")
print(f"   Nombre de locataires: {len(df_clustering)}")
display(df_clustering.head())

In [ ]:
# K-Means Clustering
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Sélection des features pour le clustering
features_clustering = ['montant_total', 'montant_moyen', 'nb_paiements', 'anciennete_jours']
X_cluster = df_clustering[features_clustering].fillna(0)

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Méthode du coude pour trouver le nombre optimal de clusters
inertias = []
K_range = range(2, 8)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualisation de la méthode du coude
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Nombre de Clusters (k)')
plt.ylabel('Inertie')
plt.title('Méthode du Coude pour Déterminer le Nombre Optimal de Clusters')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Choix de 4 clusters basé sur la méthode du coude
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df_clustering['cluster'] = kmeans.fit_predict(X_scaled)

print(f"\n✅ Clustering terminé avec {optimal_k} clusters")
print(f"   Distribution: {df_clustering['cluster'].value_counts().sort_index().to_dict()}")

In [ ]:
# Analyse des clusters
cluster_analysis = df_clustering.groupby('cluster')[features_clustering].mean()

print("=" * 80)
print("📊 ANALYSE DES CLUSTERS")
print("=" * 80)
display(cluster_analysis.style.format('{:,.2f}'))

# Interprétation des clusters
print("\n" + "=" * 80)
print("🎯 INTERPRÉTATION DES SEGMENTS")
print("=" * 80)

cluster_names = {
    0: 'Locataires Occasionnels',
    1: 'Locataires Premium',
    2: 'Locataires Fidèles',
    3: 'Nouveaux Locataires'
}

for cluster_id in range(optimal_k):
    cluster_data = cluster_analysis.loc[cluster_id]
    print(f"\n📍 Cluster {cluster_id} - {cluster_names.get(cluster_id, 'Segment')}")
    print(f"   Montant Total Moyen: {cluster_data['montant_total']:,.0f} FCFA")
    print(f"   Montant Moyen/Paiement: {cluster_data['montant_moyen']:,.0f} FCFA")
    print(f"   Nombre Moyen de Paiements: {cluster_data['nb_paiements']:.1f}")
    print(f"   Ancienneté Moyenne: {cluster_data['anciennete_jours']:.0f} jours")

In [ ]:
# Visualisation des clusters (PCA pour réduction de dimension)
from sklearn.decomposition import PCA

# PCA pour visualisation 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df_clustering['pca1'] = X_pca[:, 0]
df_clustering['pca2'] = X_pca[:, 1]

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df_clustering['pca1'], df_clustering['pca2'], 
                     c=df_clustering['cluster'], cmap='viridis', 
                     s=100, alpha=0.7, edgecolors='black')

plt.xlabel(f'Composante Principale 1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'Composante Principale 2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Segmentation des Locataires (K-Means avec PCA)')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Radar chart pour comparer les clusters
from math import pi

# Normalisation pour le radar chart
radar_data = cluster_analysis.copy()
for col in radar_data.columns:
    radar_data[col] = (radar_data[col] - radar_data[col].min()) / (radar_data[col].max() - radar_data[col].min())

categories = list(radar_data.columns)
angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for cluster_id in range(optimal_k):
    values = radar_data.loc[cluster_id].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=f'Cluster {cluster_id}')
    ax.fill(angles, values, alpha=0.1)

ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), categories)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.title('Profil des Clusters de Locataires', y=1.1)
plt.tight_layout()
plt.show()

## 📊 Partie 8 : Dashboard et Conclusions

In [ ]:
# Dashboard récapitulatif
fig = plt.figure(figsize=(20, 15))

# Titre
fig.suptitle('🏠 DASHBOARD DATA SCIENCE - CI IMMOBILIER', fontsize=20, fontweight='bold', y=0.98)

# 1. KPIs
ax1 = fig.add_subplot(3, 3, 1)
kpi_data = {
    'Total Propriétés': len(df_proprietes),
    'Total Locataires': len(df_locataires),
    'Total Paiements': len(df_paiements),
    'Revenu Total': df_paiements['montant'].sum() / 1000000  # En millions
}
ax1.axis('off')
table = ax1.table(cellText=[[k, f"{v:,.0f}" if k != 'Revenu Total' else f"{v:,.2f} M"] 
                           for k, v in kpi_data.items()],
                  colLabels=['Métrique', 'Valeur'],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 1.5)
ax1.set_title('📊 KPIs Principaux', fontsize=14, fontweight='bold', pad=20)

# 2. Prix par commune
ax2 = fig.add_subplot(3, 3, 2)
top_communes = df_proprietes.groupby('commune')['prix_loyer'].mean().sort_values(ascending=False).head(5)
ax2.barh(top_communes.index, top_communes.values, color='coral')
ax2.set_xlabel('Prix Moyen (FCFA)')
ax2.set_title('🏆 Top 5 Communes (Prix Moyen)', fontsize=14, fontweight='bold')
ax2.ticklabel_format(style='plain', axis='x')

# 3. Distribution types de logement
ax3 = fig.add_subplot(3, 3, 3)
type_dist = df_proprietes['type'].value_counts()
ax3.pie(type_dist.values, labels=type_dist.index, autopct='%1.1f%%', 
        colors=plt.cm.Set3(np.linspace(0, 1, len(type_dist))))
ax3.set_title('🏠 Types de Logements', fontsize=14, fontweight='bold')

# 4. Évolution des paiements
ax4 = fig.add_subplot(3, 3, 4)
paiements_mensuels = df_paiements.groupby(pd.to_datetime(df_paiements['date_paiement']).dt.to_period('M'))['montant'].sum()
ax4.plot(range(len(paiements_mensuels)), paiements_mensuels.values, marker='o', linewidth=2)
ax4.set_xlabel('Mois')
ax4.set_ylabel('Montant (FCFA)')
ax4.set_title('📈 Évolution des Paiements', fontsize=14, fontweight='bold')
ax4.ticklabel_format(style='plain', axis='y')

# 5. Performance modèle ML
ax5 = fig.add_subplot(3, 3, 5)
best_models = df_results.nsmallest(3, 'RMSE Test')
ax5.barh(best_models['Modèle'], best_models['RMSE Test'], color='steelblue')
ax5.set_xlabel('RMSE (FCFA)')
ax5.set_title('🤖 Top 3 Modèles de Prédiction', fontsize=14, fontweight='bold')
ax5.ticklabel_format(style='plain', axis='x')

# 6. Anomalies
ax6 = fig.add_subplot(3, 3, 6)
anomaly_counts = df_paiements['anomaly_flag'].value_counts()
colors_anomaly = ['lightgreen', 'red']
ax6.bar(anomaly_counts.index, anomaly_counts.values, color=colors_anomaly, alpha=0.7)
ax6.set_ylabel('Nombre')
ax6.set_title('⚠️ Détection d\'Anomalies', fontsize=14, fontweight='bold')
for i, v in enumerate(anomaly_counts.values):
    ax6.text(i, v + 5, str(v), ha='center')

# 7. Segments locataires
ax7 = fig.add_subplot(3, 3, 7)
cluster_dist = df_clustering['cluster'].value_counts().sort_index()
cluster_labels = [cluster_names.get(i, f'Cluster {i}') for i in cluster_dist.index]
ax7.bar(cluster_labels, cluster_dist.values, color=plt.cm.viridis(np.linspace(0, 1, len(cluster_dist))))
ax7.set_xlabel('Segment')
ax7.set_ylabel('Nombre de Locataires')
ax7.set_title('🎯 Segmentation Locataires', fontsize=14, fontweight='bold')
plt.setp(ax7.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 8. Recommandations
ax8 = fig.add_subplot(3, 3, 8)
ax8.axis('off')
recommendations = [
    '✅ Optimiser les prix à Plateau/Cocody',
    '✅ Surveiller les paiements > 1M FCFA',
    '✅ Cibler les locataires premium',
    '✅ Promouvoir Wave & Orange Money',
    '✅ Utiliser Random Forest pour les prix'
]
ax8.text(0.1, 0.9, '💡 RECOMMANDATIONS', fontsize=14, fontweight='bold', transform=ax8.transAxes)
for i, rec in enumerate(recommendations):
    ax8.text(0.1, 0.8 - i*0.12, rec, fontsize=11, transform=ax8.transAxes, va='top')

# 9. Insights
ax9 = fig.add_subplot(3, 3, 9)
ax9.axis('off')
insights = [
    f"• Prix moyen: {df_proprietes['prix_loyer'].mean():,.0f} FCFA",
    f"• Taux occupation: {(df_proprietes['statut'] == 'Loué').mean()*100:.1f}%",
    f"• Panier moyen: {df_paiements['montant'].mean():,.0f} FCFA",
    f"• Anomalies: {(df_paiements['anomaly_flag'] == 'Anomalie').mean()*100:.1f}%",
    f"• Modèle R²: {df_results['R² Test'].max():.2f}"
]
ax9.text(0.1, 0.9, '🔑 INSIGHTS CLÉS', fontsize=14, fontweight='bold', transform=ax9.transAxes)
for i, insight in enumerate(insights):
    ax9.text(0.1, 0.75 - i*0.12, insight, fontsize=11, transform=ax9.transAxes, va='top')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 🎓 Conclusion et Prochaines Étapes

### ✅ Ce que vous avez appris :

1. **Exploration de données (EDA)**
   - Charger et inspecter des datasets
   - Identifier les valeurs manquantes
   - Calculer les statistiques descriptives

2. **Visualisation de données**
   - Créer des histogrammes, boxplots, heatmaps
   - Analyser les distributions et corrélations

3. **Machine Learning - Régression**
   - Préparer les données (encodage, split)
   - Entraîner plusieurs modèles
   - Évaluer avec RMSE, MAE, R²
   - Analyser l'importance des features

4. **Détection d'anomalies**
   - Méthodes statistiques (IQR)
   - Isolation Forest (ML non-supervisé)

5. **Clustering**
   - K-Means pour la segmentation
   - Méthode du coude
   - PCA pour la visualisation

6. **Dashboarding**
   - Synthétiser les insights
   - Présenter les résultats aux stakeholders

---

### 🚀 Prochaines étapes suggérées :

1. **Améliorer le modèle de prédiction**
   - Ajouter plus de features (équipements, proximité services, etc.)
   - Tester d'autres algorithmes (XGBoost, LightGBM)
   - Faire du hyperparameter tuning

2. **Time Series Forecasting**
   - Prédire les revenus futurs
   - Détecter les tendances saisonnières

3. **Système de recommandation**
   - Recommander des propriétés aux locataires
   - Optimiser les affectations

4. **Deployment**
   - Créer une API Flask/FastAPI
   - Intégrer dans l'application web
   - Dashboard interactif avec Streamlit/Plotly Dash

5. **Data Pipeline**
   - Automatiser l'extraction depuis la BDD
   - Mettre en place des rapports automatiques

---

### 📚 Ressources pour aller plus loin :

- **Documentation officielle**:
  - [pandas](https://pandas.pydata.org/docs/)
  - [scikit-learn](https://scikit-learn.org/stable/)
  - [matplotlib](https://matplotlib.org/stable/contents.html)
  - [seaborn](https://seaborn.pydata.org/tutorial.html)

- **Cours en ligne**:
  - Coursera: Machine Learning by Andrew Ng
  - DataCamp: Data Scientist Track
  - Kaggle Learn

- **Livres**:
  - "Python for Data Analysis" by Wes McKinney
  - "Hands-On Machine Learning" by Aurélien Géron

---

### 💬 Questions ?

N'hésitez pas à expérimenter avec ce notebook, modifier les paramètres, ajouter vos propres analyses, et poser des questions !

**Bon courage dans votre apprentissage de la Data Science ! 🎉**